In [5]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image
from tqdm import tqdm
from datetime import datetime

# ===============================
# SETTINGS
# ===============================

data_path = "./combined_output"
output_folder = "./pix2pix_outputs_40"

batch_size = 4
epochs = 40
lambda_L1 = 100
image_height = 128
image_width = 256

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True

print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: NVIDIA GeForce RTX 3050 4GB Laptop GPU


In [6]:
class Pix2PixDataset(Dataset):
    def __init__(self, folder):
        self.files = os.listdir(folder)
        self.folder = folder
        self.transform = transforms.Compose([
            transforms.Resize((image_height, image_width)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.folder, self.files[idx])).convert("RGB")
        img = self.transform(img)
        w = img.shape[2]
        masked = img[:, :, :w//2]
        real = img[:, :, w//2:]
        return masked, real


dataset = Pix2PixDataset(data_path)

train_size = int(0.9 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset,
                          batch_size=batch_size,
                          shuffle=True,
                          num_workers=0)

test_loader = DataLoader(test_dataset,
                         batch_size=batch_size,
                         shuffle=False,
                         num_workers=0)

print("Dataset loaded successfully")

Dataset loaded successfully


In [7]:
def down_block(in_c, out_c, normalize=True):
    layers = [nn.Conv2d(in_c, out_c, 4, 2, 1, bias=False)]
    if normalize:
        layers.append(nn.BatchNorm2d(out_c))
    layers.append(nn.LeakyReLU(0.2))
    return nn.Sequential(*layers)

def up_block(in_c, out_c):
    return nn.Sequential(
        nn.ConvTranspose2d(in_c, out_c, 4, 2, 1, bias=False),
        nn.BatchNorm2d(out_c),
        nn.ReLU()
    )

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.d1 = down_block(3, 64, normalize=False)
        self.d2 = down_block(64, 128)
        self.d3 = down_block(128, 256)
        self.d4 = down_block(256, 512)

        self.u1 = up_block(512, 256)
        self.u2 = up_block(512, 128)
        self.u3 = up_block(256, 64)

        self.final = nn.Sequential(
            nn.ConvTranspose2d(128, 3, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, x):
        d1 = self.d1(x)
        d2 = self.d2(d1)
        d3 = self.d3(d2)
        d4 = self.d4(d3)

        u1 = self.u1(d4)
        u1 = torch.cat([u1, d3], 1)

        u2 = self.u2(u1)
        u2 = torch.cat([u2, d2], 1)

        u3 = self.u3(u2)
        u3 = torch.cat([u3, d1], 1)

        return self.final(u3)

In [8]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(6, 64, 4, 2, 1),
            nn.LeakyReLU(0.2),

            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),

            nn.Conv2d(128, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),

            nn.Conv2d(256, 1, 4, 1, 1)
        )

    def forward(self, x, y):
        return self.model(torch.cat([x, y], 1))

In [9]:
G = Generator().to(device)
D = Discriminator().to(device)

loss_GAN = nn.BCEWithLogitsLoss()
loss_L1 = nn.L1Loss()

opt_G = optim.Adam(G.parameters(), lr=0.0002, betas=(0.5,0.999))
opt_D = optim.Adam(D.parameters(), lr=0.0002, betas=(0.5,0.999))

print("Models initialized")

Models initialized


In [11]:
from tqdm import tqdm
from datetime import datetime

# -----------------------------
# RESUME FROM CHECKPOINT
# -----------------------------
start_epoch = 40
extra_epochs = 5

G.load_state_dict(torch.load("pix2pix_epoch_40.pth"))
print("Checkpoint loaded: pix2pix_epoch_40.pth")

# -----------------------------
# CREATE LOG FILE
# -----------------------------
log_file = open("training_log.txt", "a")
log_file.write("Time,Epoch,G_loss,D_loss\n")

# -----------------------------
# TRAINING LOOP
# -----------------------------
for epoch in range(start_epoch, start_epoch + extra_epochs):

    total_g = 0
    total_d = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{start_epoch+extra_epochs}")

    for masked, real in progress_bar:

        masked = masked.to(device)
        real = real.to(device)

        # -----------------------
        # GENERATOR TRAINING
        # -----------------------
        fake = G(masked)

        pred_fake = D(masked, fake)
        valid = torch.ones_like(pred_fake)

        g_loss = loss_GAN(pred_fake, valid) + lambda_L1 * loss_L1(fake, real)

        opt_G.zero_grad()
        g_loss.backward()
        opt_G.step()

        # -----------------------
        # DISCRIMINATOR TRAINING
        # -----------------------
        pred_real = D(masked, real)
        pred_fake = D(masked, fake.detach())

        fake_label = torch.zeros_like(pred_fake)

        d_loss = (loss_GAN(pred_real, valid) +
                  loss_GAN(pred_fake, fake_label)) * 0.5

        opt_D.zero_grad()
        d_loss.backward()
        opt_D.step()

        total_g += g_loss.item()
        total_d += d_loss.item()

        progress_bar.set_postfix({
            "G_loss": f"{g_loss.item():.4f}",
            "D_loss": f"{d_loss.item():.4f}"
        })

    # -----------------------
    # AVERAGE LOSSES
    # -----------------------
    avg_g = total_g / len(train_loader)
    avg_d = total_d / len(train_loader)

    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    print(f"[{current_time}] Epoch {epoch+1}  G_loss: {avg_g:.4f}  D_loss: {avg_d:.4f}")

    log_file.write(f"{current_time},{epoch+1},{avg_g:.4f},{avg_d:.4f}\n")
    log_file.flush()

    # -----------------------
    # SAVE CHECKPOINT
    # -----------------------
    torch.save(G.state_dict(), f"pix2pix_epoch_{epoch+1}.pth")

# -----------------------------
# CLOSE LOG FILE
# -----------------------------
log_file.close()
print("Training finished")

Checkpoint loaded: pix2pix_epoch_40.pth


Epoch 41/45: 100%|██████████| 2248/2248 [09:53<00:00,  3.79it/s, G_loss=5.2092, D_loss=0.4276]


[2026-03-12 23:24:58] Epoch 41  G_loss: 4.4727  D_loss: 0.6177


Epoch 42/45: 100%|██████████| 2248/2248 [08:42<00:00,  4.30it/s, G_loss=4.6817, D_loss=0.7313]


[2026-03-12 23:33:41] Epoch 42  G_loss: 4.5809  D_loss: 0.5547


...

[2026-03-12 23:37:39] Epoch 43  G_loss: 4.6955  D_loss: 0.5081


Epoch 44/45: 100%|██████████| 2248/2248 [04:26<00:00,  8.42it/s, G_loss=8.1229, D_loss=0.1233] 


[2026-03-12 23:42:06] Epoch 44  G_loss: 5.8259  D_loss: 0.2985


Epoch 45/45: 100%|██████████| 2248/2248 [05:03<00:00,  7.42it/s, G_loss=11.3887, D_loss=0.0105]

[2026-03-12 23:47:09] Epoch 45  G_loss: 6.5471  D_loss: 0.2415
Training finished


In [ ]:
os.makedirs(output_folder, exist_ok=True)

G.eval()

with torch.no_grad():
    for i, (masked, real) in enumerate(test_loader):

        masked = masked.to(device)
        real = real.to(device)
        fake = G(masked)

        masked = masked * 0.5 + 0.5
        fake = fake * 0.5 + 0.5
        real = real * 0.5 + 0.5

        for j in range(masked.size(0)):
            result = torch.cat([masked[j:j+1],
                                fake[j:j+1],
                                real[j:j+1]], dim=3)

            save_image(result, f"{output_folder}/result_{i}_{j}.png")

print("Testing Completed")

Testing Completed
